#Transform Drivers Data

1. Read bronze drivers table
1. Keep only the columns required for analytics (Drop url column)
1. Standardise column names using snake_case ( driversId → driver_id)
1. Rename columns to make them more meaningful (name → consrtuctort_name)
1. Remove duplicate records
1. Transform values of column nationality to Title Case
1. Write the transformed data to silver Drivers table

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.constructors"
silver_table = f"{catalog_name}.{silver_schema}.constructors"

In [0]:
from pyspark.sql import functions as f

**Step 1 - Read bronze constructors table**

In [0]:
constructors_df = spark.table(bronze_table)

**Step 2 - Keep only the columns required for analytics (Drop url column)**

In [0]:
constructors_dropped_df = constructors_df.drop("url")

**Step 3&4 -Standardise column names**
- Standardise column names using snake_case (raceName → race_name, circuitId → circuit_id)
- Rename columns to make them more meaningful (date → race_date)

In [0]:
constructors_renamed_df = (
    constructors_dropped_df
        .withColumnsRenamed({
            "constructorId": "constructor_id",
            "name": "constructor_name"
        })
)

In [0]:
display(constructors_renamed_df)

 **Step 5 - Remove duplicate records**

In [0]:
constructors_distinct_df = constructors_renamed_df.dropDuplicates(["constructor_id"])


In [0]:
display(constructors_distinct_df)


**Step 6 - Transform values of column nationality to Title Case**

In [0]:
constructors_final_df = (
    constructors_distinct_df
        .withColumn("nationality", f.initcap(f.col("nationality")))
)

In [0]:
display(constructors_final_df)



Step 7 - Write the transformed data to silver constructors table

In [0]:
(
    constructors_final_df
        .write
        .format("delta")
        .mode("overwrite")
       .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))